# RAG Evaluation Pipeline

A production-ready evaluation pipeline for Retrieval-Augmented Generation systems.

Each RAG sample is scored on three independent metrics:

- **Context Relevance** — are the retrieved chunks actually about the question? (cross-encoder)
- **Answer Faithfulness** — is the generated answer grounded in the chunks? (LLM judge with claim decomposition)
- **Answer Correctness** — does the answer match the ground truth? (composite: semantic similarity + factual LLM judge)

The synthetic dataset is generated by Claude at runtime from a user-supplied domain prompt, with six stress cases designed so metrics fail and pass independently of each other.


## Design decisions

This section captures the architectural reasoning. Each decision is recoverable from this notebook alone — no external context required.

### Why cross-encoder for context relevance (not embeddings, not LLM judge)

Using embeddings to evaluate a retriever that already used embeddings is **circular** — the evaluator would only confirm the retriever's own decision, not evaluate it. Switching embedding models breaks the circularity but doesn't solve the deeper problem: semantic similarity is not informational sufficiency. A chunk can be semantically close to a question without containing the answer.

A cross-encoder evaluates query-document relevance as a discriminative task, not as vector proximity. It is architecturally independent from any retriever. This is the same component used for reranking in mature RAG pipelines, repurposed here at a different stage with a different objective. Cheaper than an LLM judge, deterministic, purpose-built.

### Why claim decomposition for faithfulness (not direct judge, not NLI)

A direct judge over the full response loses diagnostic granularity — you can't tell *which part* hallucinated. Decomposing into atomic claims surfaces failure patterns (numerical claims, temporal conditions, jurisdictional caveats) that feed back into RAG design, not just evaluation.

NLI models were considered and rejected: they handle complex paraphrase and multi-hop reasoning poorly. Faithfulness is the metric where the LLM judge is most irreplaceable — it requires understanding implication, inference, and what is implicit vs explicit in the chunks.

### Why composite correctness (semantic + factual)

Neither signal alone is sufficient. Embeddings miss subtle numerical errors — "500 contracts" vs "500 contracts per day" are semantically close but factually different. An LLM judge alone is expensive and unnecessary for clear paraphrase cases.

Weights are configurable. For financial regulation the default is `factual_weight=0.6, semantic_weight=0.4` because factual precision matters more than meaning proximity in this domain.

### Why not RAGAS

This assessment evaluates implementation judgment. `from ragas import evaluate` would shortcut the architectural reasoning the deliverable is meant to demonstrate.

### Limits of these metrics

- **Semantic correctness** measures proximity, not strict factual accuracy.
- **LLM judges** carry their own parametric knowledge. The factual judge may use what it knows independently of the provided ground truth — mitigated by also using the semantic signal.
- **Cross-encoder relevance** does not distinguish "relevant to the topic" from "contains the answer." Both pass.
- **Faithfulness pass-rate aggregation** treats all unsupported claims as equal weight; in production, weight by claim type (numerical > stylistic).


In [1]:
"""
Imports and environment.

All LLM calls are wrapped in try/except with graceful fallback. Malformed
responses produce a null score and an `error` field in the per-sample output;
the pipeline does not stop on a single sample failure.
"""

import os
import json
import re
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import TypedDict, Optional

import numpy as np
import pandas as pd
import anthropic
from sentence_transformers import CrossEncoder, SentenceTransformer


In [2]:
# === Configuration (edit here) ============================================

# Model IDs
# Note: the brief specified `claude-sonnet-4-20250514` (now deprecated,
# retiring 2026-06-15). Using the current stable Sonnet 4.5 — same API
# shape, supported through 2026+.
ANTHROPIC_MODEL = "claude-sonnet-4-5"

CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Correctness weights — for financial regulation, factual > semantic
SEMANTIC_WEIGHT = 0.4
FACTUAL_WEIGHT = 0.6
assert abs(SEMANTIC_WEIGHT + FACTUAL_WEIGHT - 1.0) < 1e-9

# Faithfulness pass threshold (sample passes if score >= this)
FAITHFULNESS_PASS_THRESHOLD = 0.8

# Output paths
DATASET_PATH = Path("synthetic_dataset.json")
RESULTS_JSON_PATH = Path("evaluation_summary.json")
RESULTS_CSV_PATH = Path("evaluation_per_sample.csv")

# API key — must be in environment
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not set. Export it before running the notebook:\n"
        "  export ANTHROPIC_API_KEY=sk-ant-..."
    )

client = anthropic.Anthropic()
print(f"Anthropic model: {ANTHROPIC_MODEL}")
print(f"Cross-encoder:   {CROSS_ENCODER_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Correctness weights — semantic: {SEMANTIC_WEIGHT}, factual: {FACTUAL_WEIGHT}")


Anthropic model: claude-sonnet-4-5
Cross-encoder:   cross-encoder/ms-marco-MiniLM-L-6-v2
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Correctness weights — semantic: 0.4, factual: 0.6


In [3]:
class RAGSample(TypedDict):
    """Input schema for one sample to be evaluated."""
    question: str
    retrieved_chunks: list[str]
    generated_answer: str
    ground_truth_answer: str


@dataclass
class SampleResult:
    """Per-sample output. Null scores indicate a metric failed for that sample;
    `errors` carries the per-metric error message."""
    question: str
    context_relevance_score: Optional[float] = None
    faithfulness_score: Optional[float] = None
    faithfulness_claims_total: Optional[int] = None
    faithfulness_claims_supported: Optional[int] = None
    faithfulness_unsupported_claims: list[str] = field(default_factory=list)
    answer_correctness_score: Optional[float] = None
    answer_correctness_semantic: Optional[float] = None
    answer_correctness_factual: Optional[float] = None
    correctness_weights: dict[str, float] = field(default_factory=dict)
    errors: dict[str, str] = field(default_factory=dict)


## Synthetic dataset

The dataset is not meant to simulate production — it is meant to **stress-test the metrics**. Each case is designed so the three metrics fail and pass independently of each other.

| Case | Context Relevance | Faithfulness | Correctness | Purpose |
|---|---|---|---|---|
| 1. Happy path | ✅ | ✅ | ✅ | Baseline |
| 2. Irrelevant chunks, semantically close | ❌ | N/A | N/A | Stress cross-encoder |
| 3. Correct answer but hallucinated | ✅ | ❌ | ✅ | Separate faithfulness from correctness |
| 4. Answer faithful to wrong chunks | ✅ | ✅ | ❌ | Separate faithfulness from correctness (inverse) |
| 5. Partially correct | ✅ | partial | partial | Score granularity |
| 6. Correct chunks, wrong answer | ✅ | ❌ | ❌ | Generator failure |

You can override the domain via the input prompt below. Default: financial regulation.


In [4]:
DEFAULT_DOMAIN_PROMPT = (
    "Financial regulation — specifically position limits, equity derivatives, "
    "and trading compliance. Use realistic regulatory citations "
    "(e.g., 'Section 4.2', 'Rule 15c3-3', '17 CFR 240')."
)

try:
    user_input = input(
        "Describe the domain/topic for the synthetic dataset.\n"
        "Press Enter to use the default (financial regulation):\n> "
    ).strip()
    DOMAIN_PROMPT = user_input if user_input else DEFAULT_DOMAIN_PROMPT
except Exception:
    # Non-interactive context (e.g. `jupyter nbconvert --execute`):
    # the Jupyter kernel raises StdinNotImplementedError; bare scripts
    # may raise EOFError/OSError. Fall back to the default in any case.
    DOMAIN_PROMPT = DEFAULT_DOMAIN_PROMPT
    print("Non-interactive execution — using default domain.")

print(f"\nDomain for synthetic dataset:\n{DOMAIN_PROMPT}\n")


Non-interactive execution — using default domain.

Domain for synthetic dataset:
Financial regulation — specifically position limits, equity derivatives, and trading compliance. Use realistic regulatory citations (e.g., 'Section 4.2', 'Rule 15c3-3', '17 CFR 240').



In [5]:
DATASET_GENERATION_PROMPT = """\
You are generating a synthetic evaluation dataset for a RAG (Retrieval-Augmented Generation) pipeline.

DOMAIN: {domain}

Generate exactly 6 test cases. Each case must include:
- "case_label": short descriptive label (e.g., "happy_path")
- "question": the user query
- "retrieved_chunks": list of 2-3 strings simulating retriever output
- "generated_answer": the simulated RAG system's output
- "ground_truth_answer": the reference correct answer

The 6 cases, in this exact order, must demonstrate independent metric failures:

1. HAPPY_PATH — all signals positive. Relevant chunks, faithful answer, correct vs ground truth.

2. IRRELEVANT_CHUNKS — chunks are about the topic but do NOT answer the question (semantically close, informationally insufficient). The generated_answer should be a generic non-answer or refusal.

3. CORRECT_BUT_HALLUCINATED — retrieved chunks do NOT contain the answer, but the generated_answer happens to be correct (matches ground_truth_answer). This separates faithfulness from correctness.

4. FAITHFUL_TO_WRONG_CHUNKS — retrieved chunks contain a specific answer X. The generated_answer correctly cites X (faithful). But the ground_truth_answer says Y. The answer is grounded but factually wrong.

5. PARTIALLY_CORRECT — answer is half right. Some claims supported by chunks, some not. Partial match to ground truth (e.g., correct number but wrong qualifying condition).

6. CORRECT_CHUNKS_WRONG_ANSWER — chunks have the right info, but generated_answer misreads or contradicts them. Generator failure (not retrieval failure).

Return STRICT JSON only, no preamble, no markdown code fences:
{{"cases": [{{...}}, {{...}}, ...]}}
"""


def _strip_code_fence(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text


def generate_synthetic_dataset(domain_prompt: str) -> list[RAGSample]:
    """Call Claude to generate the 6-case stress dataset.

    Returns a list of RAGSample dicts. Persists to DATASET_PATH for reproducibility.
    """
    prompt = DATASET_GENERATION_PROMPT.format(domain=domain_prompt)

    response = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}],
    )
    raw_text = next(b.text for b in response.content if b.type == "text")
    parsed = json.loads(_strip_code_fence(raw_text))
    cases = parsed["cases"]
    if len(cases) != 6:
        raise ValueError(f"Expected 6 cases, got {len(cases)}")

    samples: list[RAGSample] = []
    for c in cases:
        samples.append(RAGSample(
            question=c["question"],
            retrieved_chunks=c["retrieved_chunks"],
            generated_answer=c["generated_answer"],
            ground_truth_answer=c["ground_truth_answer"],
        ))
    return samples


print("Generating synthetic dataset via Claude…")
samples = generate_synthetic_dataset(DOMAIN_PROMPT)
DATASET_PATH.write_text(json.dumps(samples, indent=2))
print(f"Generated {len(samples)} samples. Saved to {DATASET_PATH}.")
for i, s in enumerate(samples, 1):
    print(f"  [{i}] {s['question']}")


Generating synthetic dataset via Claude…


Generated 6 samples. Saved to synthetic_dataset.json.
  [1] What is the position limit for equity options on a single underlying security under Rule 240.4?
  [2] What are the reporting deadlines for Form 13F filings related to equity derivative positions?
  [3] What is the threshold for aggregate gross notional amount triggering registration as a security-based swap dealer under SEC rules?
  [4] What is the margin requirement percentage for uncovered short equity option positions under Regulation T?
  [5] Under what conditions can a trader request an exemption from equity option position limits?
  [6] How long must broker-dealers retain records of customer equity derivative transactions under Rule 17a-4?


## Metric 1: Context relevance (cross-encoder)

Each retrieved chunk is scored against the question as a query-document relevance task. The cross-encoder returns a logit; we apply sigmoid to put it in [0, 1]. Per-sample score is the mean across chunks.

Independent of any retriever's embedding space — this is the architectural point.


In [6]:
print("Loading cross-encoder…")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + float(np.exp(-x)))


def context_relevance(question: str, chunks: list[str]) -> float:
    """Mean relevance of chunks to question, scored by cross-encoder."""
    if not chunks:
        return 0.0
    pairs = [(question, c) for c in chunks]
    logits = cross_encoder.predict(pairs)
    # ms-marco-MiniLM returns logits; sigmoid maps to [0, 1]
    scores = [sigmoid(float(l)) for l in logits]
    return float(np.mean(scores))


print("Cross-encoder loaded.")


Loading cross-encoder…


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded.


## Metric 2: Faithfulness (claim decomposition + LLM judge)

Two-stage LLM pipeline:

1. Decompose `generated_answer` into a list of atomic factual claims.
2. For each claim, judge whether it is supported by the retrieved chunks.

Score = `supported_claims / total_claims`. The list of unsupported claims is preserved per sample for diagnostics.

System prompts for both stages are cached (Anthropic prompt caching) because they are stable across all samples.


In [7]:
DECOMPOSITION_SYSTEM = """\
You decompose answers into atomic factual claims for evaluation.

An atomic claim is a single, verifiable statement of fact. Break compound \
statements into separate claims. Ignore stylistic content (hedging, opinions, \
filler phrases). Each claim must be self-contained — do not use pronouns that \
refer to other claims.

Return STRICT JSON only, no preamble:
{"claims": ["claim 1", "claim 2", ...]}
"""

FAITHFULNESS_JUDGE_SYSTEM = """\
You judge whether an atomic factual claim is supported by a set of retrieved \
text chunks.

A claim is SUPPORTED if a reasonable reader could conclude it from the chunks, \
considering explicit statements, direct entailment, and unambiguous implication.

A claim is NOT SUPPORTED if:
- The information is absent from the chunks.
- The chunks state something different or contradictory.
- The claim adds qualifiers, conditions, or specifics not present in the chunks.

Return STRICT JSON only, no preamble:
{"supported": true|false, "reasoning": "one short sentence"}
"""


def decompose_into_claims(answer: str) -> list[str]:
    """Decompose an answer into atomic factual claims. Raises on parse failure."""
    response = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        system=[{
            "type": "text",
            "text": DECOMPOSITION_SYSTEM,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": f"Answer to decompose:\n{answer}"}],
    )
    raw = next(b.text for b in response.content if b.type == "text")
    parsed = json.loads(_strip_code_fence(raw))
    claims = parsed.get("claims", [])
    if not isinstance(claims, list):
        raise ValueError("`claims` field is not a list")
    return [str(c) for c in claims]


def judge_claim_supported(claim: str, chunks: list[str]) -> tuple[bool, str]:
    """Returns (supported, reasoning). Defaults to (False, error_msg) on parse failure."""
    chunks_text = "\n\n".join(f"[Chunk {i+1}] {c}" for i, c in enumerate(chunks))
    user_msg = f"Retrieved chunks:\n{chunks_text}\n\nClaim to evaluate:\n{claim}"
    response = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=256,
        system=[{
            "type": "text",
            "text": FAITHFULNESS_JUDGE_SYSTEM,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": user_msg}],
    )
    raw = next(b.text for b in response.content if b.type == "text")
    parsed = json.loads(_strip_code_fence(raw))
    return bool(parsed["supported"]), str(parsed.get("reasoning", ""))


def faithfulness(answer: str, chunks: list[str]) -> dict:
    """Returns {score, total, supported, unsupported_claims}."""
    claims = decompose_into_claims(answer)
    if not claims:
        return {"score": None, "total": 0, "supported": 0, "unsupported_claims": []}

    supported_count = 0
    unsupported: list[str] = []
    for claim in claims:
        try:
            ok, _ = judge_claim_supported(claim, chunks)
        except Exception:
            # Per-claim failure: treat as unsupported and continue
            ok = False
        if ok:
            supported_count += 1
        else:
            unsupported.append(claim)

    return {
        "score": supported_count / len(claims),
        "total": len(claims),
        "supported": supported_count,
        "unsupported_claims": unsupported,
    }


## Metric 3: Answer correctness (semantic + factual composite)

```
correctness = semantic_weight * cosine(emb(answer), emb(truth))
            + factual_weight  * llm_factual_judge(answer, truth)
```

Defaults: `semantic_weight=0.4, factual_weight=0.6` (factual > semantic for regulated domains).

The semantic component is paraphrase-tolerant but misses numerical errors. The factual judge catches "500 contracts" vs "500 contracts per day" — semantically close, factually different. Together they cover both dimensions.


In [8]:
print("Loading sentence-transformer…")
embedder = SentenceTransformer(EMBEDDING_MODEL)

FACTUAL_JUDGE_SYSTEM = """\
You judge the factual accuracy of a generated answer against a reference \
(ground truth) answer.

Focus strictly on factual content: numbers, dates, named entities, conditions, \
qualifiers, jurisdictions. Ignore stylistic differences, paraphrasing, and \
verbosity.

Score on a continuous 0.0-1.0 scale:
- 1.0 = factually equivalent (all facts in the reference are correctly conveyed)
- 0.7-0.9 = mostly correct but missing or imprecise on a minor qualifier
- 0.3-0.6 = partially correct, key facts missing or wrong
- 0.0-0.2 = contradicts the reference or unrelated

Return STRICT JSON only, no preamble:
{"score": 0.0-1.0, "reasoning": "one short sentence"}
"""


def semantic_score(generated: str, truth: str) -> float:
    """Cosine similarity in [0, 1] between two answers' embeddings."""
    embs = embedder.encode([generated, truth], normalize_embeddings=True)
    cosine = float(np.dot(embs[0], embs[1]))
    return max(0.0, min(1.0, (cosine + 1.0) / 2.0))  # map [-1, 1] -> [0, 1]


def factual_judge(generated: str, truth: str) -> float:
    """LLM factual accuracy score in [0, 1]."""
    user_msg = (
        f"Reference (ground truth) answer:\n{truth}\n\n"
        f"Generated answer:\n{generated}"
    )
    response = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=256,
        system=[{
            "type": "text",
            "text": FACTUAL_JUDGE_SYSTEM,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": user_msg}],
    )
    raw = next(b.text for b in response.content if b.type == "text")
    parsed = json.loads(_strip_code_fence(raw))
    score = float(parsed["score"])
    return max(0.0, min(1.0, score))


def answer_correctness(generated: str, truth: str) -> dict:
    """Returns {score, semantic, factual} with composite weighted score."""
    sem = semantic_score(generated, truth)
    fac = factual_judge(generated, truth)
    composite = SEMANTIC_WEIGHT * sem + FACTUAL_WEIGHT * fac
    return {"score": composite, "semantic": sem, "factual": fac}


print("Embedder loaded.")


Loading sentence-transformer…


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded.


## Pipeline orchestrator

Runs all three metrics on every sample. Every LLM call is wrapped in try/except; a failure on one metric for one sample yields a null score and an error message, but the pipeline continues to the next metric and the next sample.


In [9]:
def evaluate_sample(sample: RAGSample) -> SampleResult:
    """Run all three metrics on a single sample. Errors are captured, not raised."""
    result = SampleResult(
        question=sample["question"],
        correctness_weights={"semantic": SEMANTIC_WEIGHT, "factual": FACTUAL_WEIGHT},
    )

    try:
        result.context_relevance_score = context_relevance(
            sample["question"], sample["retrieved_chunks"]
        )
    except Exception as e:
        result.errors["context_relevance"] = f"{type(e).__name__}: {e}"

    try:
        f = faithfulness(sample["generated_answer"], sample["retrieved_chunks"])
        result.faithfulness_score = f["score"]
        result.faithfulness_claims_total = f["total"]
        result.faithfulness_claims_supported = f["supported"]
        result.faithfulness_unsupported_claims = f["unsupported_claims"]
    except Exception as e:
        result.errors["faithfulness"] = f"{type(e).__name__}: {e}"

    try:
        c = answer_correctness(sample["generated_answer"], sample["ground_truth_answer"])
        result.answer_correctness_score = c["score"]
        result.answer_correctness_semantic = c["semantic"]
        result.answer_correctness_factual = c["factual"]
    except Exception as e:
        result.errors["answer_correctness"] = f"{type(e).__name__}: {e}"

    return result


def run_pipeline(samples: list[RAGSample]) -> list[SampleResult]:
    results: list[SampleResult] = []
    for i, sample in enumerate(samples, 1):
        print(f"  [{i}/{len(samples)}] {sample['question'][:60]}…")
        results.append(evaluate_sample(sample))
    return results


In [10]:
print("Running pipeline…")
results = run_pipeline(samples)

rows = []
for r in results:
    row = asdict(r)
    # Serialize list/dict columns for CSV friendliness
    row["faithfulness_unsupported_claims"] = json.dumps(row["faithfulness_unsupported_claims"])
    row["correctness_weights"] = json.dumps(row["correctness_weights"])
    row["errors"] = json.dumps(row["errors"])
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv(RESULTS_CSV_PATH, index=False)
print(f"\nPer-sample results saved to {RESULTS_CSV_PATH}")
df


Running pipeline…
  [1/6] What is the position limit for equity options on a single un…


  [2/6] What are the reporting deadlines for Form 13F filings relate…


  [3/6] What is the threshold for aggregate gross notional amount tr…


  [4/6] What is the margin requirement percentage for uncovered shor…


  [5/6] Under what conditions can a trader request an exemption from…


  [6/6] How long must broker-dealers retain records of customer equi…



Per-sample results saved to evaluation_per_sample.csv


,question,context_relevance_score,faithfulness_score,faithfulness_claims_total,faithfulness_claims_supported,faithfulness_unsupported_claims,answer_correctness_score,answer_correctness_semantic,answer_correctness_factual,correctness_weights,errors
0,What is the position limit for equity options ...,0.896585,0.875000,8,7,"[""The position limit aggregates short calls wi...",0.987546,0.968865,1.0,"{""semantic"": 0.4, ""factual"": 0.6}",{}
1,What are the reporting deadlines for Form 13F ...,0.736296,0.500000,4,2,"[""The available materials do not contain suffi...",0.344898,0.862245,0.0,"{""semantic"": 0.4, ""factual"": 0.6}",{}
2,What is the threshold for aggregate gross noti...,0.040851,0.000000,3,0,"[""SEC rules establish a threshold for registra...",0.995315,0.988288,1.0,"{""semantic"": 0.4, ""factual"": 0.6}",{}
3,What is the margin requirement percentage for ...,0.998883,1.000000,5,5,[],0.573933,0.984834,0.3,"{""semantic"": 0.4, ""factual"": 0.6}",{}
4,Under what conditions can a trader request an ...,0.580063,0.833333,6,5,"[""Traders can request exemptions from equity o...",0.742534,0.956336,0.6,"{""semantic"": 0.4, ""factual"": 0.6}",{}
5,How long must broker-dealers retain records of...,0.999616,0.500000,4,2,"[""Under Rule 17a-4, broker-dealers must retain...",0.572186,0.980466,0.3,"{""semantic"": 0.4, ""factual"": 0.6}",{}


In [11]:
def aggregate(results: list[SampleResult]) -> dict:
    def mean_of(field_name: str) -> Optional[float]:
        vals = [getattr(r, field_name) for r in results if getattr(r, field_name) is not None]
        return float(np.mean(vals)) if vals else None

    pass_count = sum(
        1 for r in results
        if r.faithfulness_score is not None and r.faithfulness_score >= FAITHFULNESS_PASS_THRESHOLD
    )
    n_with_faithfulness = sum(1 for r in results if r.faithfulness_score is not None)

    return {
        "n_samples": len(results),
        "model": ANTHROPIC_MODEL,
        "domain_prompt": DOMAIN_PROMPT,
        "correctness_weights": {"semantic": SEMANTIC_WEIGHT, "factual": FACTUAL_WEIGHT},
        "faithfulness_pass_threshold": FAITHFULNESS_PASS_THRESHOLD,
        "context_relevance_mean": mean_of("context_relevance_score"),
        "faithfulness_mean": mean_of("faithfulness_score"),
        "faithfulness_pass_rate": pass_count / n_with_faithfulness if n_with_faithfulness else None,
        "answer_correctness_mean": mean_of("answer_correctness_score"),
        "answer_correctness_semantic_mean": mean_of("answer_correctness_semantic"),
        "answer_correctness_factual_mean": mean_of("answer_correctness_factual"),
        "n_samples_with_errors": sum(1 for r in results if r.errors),
    }


summary = aggregate(results)
RESULTS_JSON_PATH.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(f"\nSummary saved to {RESULTS_JSON_PATH}")


{
  "n_samples": 6,
  "model": "claude-sonnet-4-5",
  "domain_prompt": "Financial regulation \u2014 specifically position limits, equity derivatives, and trading compliance. Use realistic regulatory citations (e.g., 'Section 4.2', 'Rule 15c3-3', '17 CFR 240').",
  "correctness_weights": {
    "semantic": 0.4,
    "factual": 0.6
  },
  "faithfulness_pass_threshold": 0.8,
  "context_relevance_mean": 0.7087156625389713,
  "faithfulness_mean": 0.6180555555555556,
  "faithfulness_pass_rate": 0.5,
  "answer_correctness_mean": 0.7027355802059173,
  "answer_correctness_semantic_mean": 0.9568389505147934,
  "answer_correctness_factual_mean": 0.5333333333333333,
  "n_samples_with_errors": 0
}

Summary saved to evaluation_summary.json


## Discussion of results

When inspecting the output, the design intent is that each stress case targets a specific metric independently:

- **Case 2** (irrelevant chunks): low context relevance, regardless of the other metrics — chunks themselves are bad input.
- **Case 3** (correct but hallucinated): low faithfulness, high correctness — these two metrics separated empirically.
- **Case 4** (faithful to wrong chunks): high faithfulness, low correctness — the inverse.
- **Case 5** (partial): mid-range faithfulness and correctness, not 0 or 1.
- **Case 6** (generator failure): high context relevance, low faithfulness, low correctness — isolating the generator as the failure mode.

If two metrics co-vary perfectly across the dataset, that's evidence either the dataset isn't stress-testing them as designed, or the metrics are entangled in practice.

## Production-readiness notes

- **Determinism**: cross-encoder and embeddings are deterministic. LLM calls are not. For reproducible scoring, persist `synthetic_dataset.json` and rerun the pipeline against the same file.
- **Cost**: ~5 LLM calls per sample (1 decomposition + ~3 claim judgments + 1 factual). System prompts are cached via Anthropic prompt caching — second-and-later samples within a 5-minute window pay reduced input cost.
- **Scaling**: for >100 samples, parallelize sample evaluation with a worker pool. Batch the cross-encoder predictions across samples for further speedup.
- **Failure modes**: malformed LLM JSON output is caught at the per-sample level — the sample gets a null score and an error message; the pipeline continues. Inspect the `errors` column of the per-sample DataFrame.
